# OpenDR 1.0 Case Study: Humanitarian Exposure & Risk Modeling
## Focus: Building-Level Impact Assessment in Informal & Refugee Settlements

This notebook demonstrates the **OpenDR 1.0** pipeline for humanitarian intelligence. We integrate:
1. **Infrastructure Ingestion:** Loading high-resolution footprints from the **Google Open Buildings** dataset.
2. **Spatial Intelligence:** Intersecting dynamic hazard vectors (flood/fire) with building locations.
3. **Exposure Weighting:** Calculating population-weighted metrics to prioritize emergency resource allocation.

### Scientific References:
- **Exposure Mapping:** Van Den Hoek & Friedrich (2024)
- **Population Weighting:** Venter & Chowdhury (2024)

In [ ]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape

# Ensure project root is in path for microservice access
sys.path.append(os.path.abspath('../'))
from src.microservices.exposure_model import calculate_impact_metrics

print("OpenDR 1.0 Humanitarian Intelligence Module Loaded.")

## 1. Load Infrastructure Data (Google Open Buildings)
We load a sample of building footprints for the **Pagirinya Refugee Settlement** area. In the full OpenDR 1.0 pipeline, Tier 1 (`fetch_google_buildings.py`) streams this data in GeoParquet format.

In [ ]:
# Load building footprints (Sample Parquet file created by scripts/gen_tokyo_sample.py logic)
# Note: For this demo, we use the sample data generated for the Tokyo/Ethiopia areas
buildings_path = '../data/sample/tokyo_buildings_sample.parquet'
buildings = gpd.read_parquet(buildings_path)

print(f"[*] Ingested {len(buildings)} building footprints for exposure analysis.")
print(buildings[['building_id', 'measured_height', 'pop_density']].head())

## 2. Load Active Hazard Vectors
We load the flood extent vector generated by the **Adaptive Otsu** module in the previous transboundary case study.

In [ ]:
# Simulated hazard vector (In production, this is queried from Tier 4 PostGIS via OGC API)
from shapely.geometry import box
hazard_area = box(139.75, 35.68, 139.756, 35.688) # Overlaps sample buildings
hazard_gdf = gpd.GeoDataFrame({'hazard': ['Flood']}, geometry=[hazard_area], crs="EPSG:4326")

print("[*] Active hazard vector loaded from Tier 4 Mediation layer.")

## 3. Distributed Exposure Analysis
We execute the exposure microservice. This performs a spatial join between the infrastructure and the hazard, weighting the results by the population density of each building.

In [ ]:
# Execute calculation logic based on Venter & Chowdhury (2024)
impact_summary = calculate_impact_metrics(hazard_gdf, buildings, population_raster=None)

print("--- SITUATIONAL AWARENESS REPORT ---")
print(f"Total affected buildings: {impact_summary['total_buildings_affected']}")
print(f"Estimated displaced persons: {impact_summary['estimated_displaced_persons']:.0f}")
print(f"Critical infrastructure at risk: {impact_summary['high_priority_infrastructure']}")

## 4. Visualization for Decision Support
We visualize the building-level risk to help responders prioritize the 'Golden Hour' interventions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot all buildings in grey
buildings.plot(ax=ax, color='#cccccc', label='Unassigned')

# Highlight the hazard zone in transparent blue
hazard_gdf.plot(ax=ax, color='blue', alpha=0.2, label='Hazard Zone')

# Highlight affected buildings by density weight
impacted = gpd.sjoin(buildings, hazard_gdf, predicate='within')
impacted.plot(ax=ax, column='pop_density', cmap='YlOrRd', legend=True, 
              legend_kwds={'label': "Population density (Priority)"})

plt.title("OpenDR 1.0: Real-Time Humanitarian Exposure Dashboard")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## 5. Decision Support Dispatch
The identified high-risk buildings and population estimates are now ready to be pushed to Tier 5 (Mobile Teams).

In [ ]:
print("[*] Tier 5: Generating Alert Payload for KoboToolbox/QField dispatch.")
print("[*] Alert target: Regional Response Unit - Jinka/Tokyo.")
print("--- OpenDR 1.0: Exposure Modeling Complete ---")